# Reproduce Frozen Run82

This notebook reproduces the frozen Run82 summarization and Round-2 preservation evaluation used in Paper 2A. It is a public, reproducibility-focused extraction of the authoritative implementation.

**Frozen configuration**

- Historical seed: **42**
- Topic weight α: **0.30**
- Pattern weight β: **0.70**
- Redundancy weight δ: **1.00**
- Summary budget: **15 sentences**
- Requested topics: **25**
- Evaluation split: **Multi-News test**

The notebook verifies the frozen validation artifacts before evaluating the held-out test split. Test results are not used for parameter selection or reselection. Public outputs contain aggregate and per-cluster numeric metrics only; source articles, reference summaries, and generated summaries are not written to disk.

In [ ]:
# Setup
import os, sys, subprocess, json, hashlib, gc, time, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "2")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

required = {
    "datasets": "datasets>=3,<5",
    "spacy": "spacy>=3.7,<4",
    "mlxtend": "mlxtend>=0.23,<1",
    "rouge_score": "rouge-score==0.1.2",
    "sklearn": "scikit-learn>=1.3,<2",
    "tqdm": "tqdm>=4.66,<5",
}
missing_specs = []
for module, spec in required.items():
    try:
        __import__(module)
    except Exception:
        missing_specs.append(spec)
if missing_specs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_specs])

model_check = subprocess.run(
    [sys.executable, "-c", "import en_core_web_sm"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if model_check.returncode != 0:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"])

# Repository root: run from the repository root, or set PAPER2A_REPO_ROOT.
REPO_ROOT = Path(os.environ.get("PAPER2A_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "validation").exists() and (REPO_ROOT.parent / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent

VALIDATION_DIR = REPO_ROOT / "validation"
OUT = REPO_ROOT / "reproduction_outputs" / "run82"
OUT.mkdir(parents=True, exist_ok=True)

DETAIL_PATH = OUT / "run82_test_cluster_metrics.csv"
RESULT_PATH = OUT / "run82_test_aggregate.csv"
PROVENANCE_PATH = OUT / "dataset_provenance_test.json"
STATUS_PATH = OUT / "run82_reproduction_status.json"

print("Repository root:", REPO_ROOT)
print("Output directory:", OUT)

In [ ]:
# Verify frozen validation artifacts
EXPECTED_SHA = "9ef9796b30ed6695f501bd330547b8c9b16c00abfe7ad1675d91b99b2a27a29a"
EXPECTED_RUN_ORDER = 82
EXPECTED = {
    "alpha_topic": 0.30,
    "beta_pattern": 0.70,
    "delta_redundancy": 1.00,
    "summary_sentence_budget": 15,
    "n_topics": 25,
}

status_file = VALIDATION_DIR / "ROUND1_FINALIZATION_STATUS.json"
frozen_file = VALIDATION_DIR / "PAPER2_FROZEN_VALIDATION_SELECTION.json"
master_file = VALIDATION_DIR / "single_scenario_results_validation_merged_120.csv"

for p in [status_file, frozen_file, master_file]:
    if not p.exists():
        raise FileNotFoundError(f"Required validation artifact not found: {p}")

ROUND1_STATUS = json.loads(status_file.read_text(encoding="utf-8"))
FROZEN = json.loads(frozen_file.read_text(encoding="utf-8"))
VALIDATION_MASTER = pd.read_csv(master_file)

canonical = VALIDATION_MASTER.drop(columns=["source_file"], errors="ignore")
actual_sha = hashlib.sha256(canonical.to_csv(index=False).encode("utf-8")).hexdigest()

errors = []
if len(VALIDATION_MASTER) != 120:
    errors.append(f"Expected 120 validation rows; found {len(VALIDATION_MASTER)}")
if VALIDATION_MASTER["configuration_id"].astype(str).nunique() != 120:
    errors.append("Validation table does not contain 120 unique configuration IDs")
if actual_sha != EXPECTED_SHA:
    errors.append(f"Validation SHA-256 mismatch: {actual_sha}")
if FROZEN.get("validation_results_sha256") != EXPECTED_SHA:
    errors.append("Frozen-selection record has an unexpected validation SHA-256")
if int(FROZEN.get("selected_validation_run_order", -1)) != EXPECTED_RUN_ORDER:
    errors.append("Frozen selected run is not Run82")
if FROZEN.get("selection_split") != "validation":
    errors.append("Frozen selection split is not validation")
if FROZEN.get("definitive_evaluation_split") != "test":
    errors.append("Definitive evaluation split is not test")
if not FROZEN.get("frozen_before_test", False):
    errors.append("Frozen-selection record does not confirm freezing before test evaluation")

for key, expected in EXPECTED.items():
    actual = FROZEN.get(key)
    if isinstance(expected, float):
        if actual is None or not np.isclose(float(actual), expected, rtol=0, atol=1e-12):
            errors.append(f"{key}: expected {expected}, got {actual}")
    elif actual is None or int(actual) != expected:
        errors.append(f"{key}: expected {expected}, got {actual}")

if errors:
    raise RuntimeError("Frozen-artifact verification failed:\n- " + "\n- ".join(errors))

ALPHA = 0.30
BETA = 0.70
DELTA = 1.00
SUMMARY_BUDGET = 15
N_TOPICS = 25
SEED = 42
DATASET_SPLIT = "test"

print("Frozen Run82 verified")
print("Validation SHA-256:", actual_sha)
print("Parameters:", EXPECTED)
print("Historical seed:", SEED)

In [ ]:
# Load the held-out Multi-News test split
from datasets import load_dataset

DATASET_REPO = "Awesome075/multi_news_parquet"

def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " ")).strip()

raw_test = load_dataset(DATASET_REPO, split="test")
summary_data = []
excluded = 0

for idx, row in enumerate(raw_test):
    raw_document = clean_text(row.get("document", ""))
    reference = clean_text(row.get("summary", ""))
    documents = [clean_text(x) for x in raw_document.split("|||||") if clean_text(x)]
    if not documents or not reference:
        excluded += 1
        continue
    summary_data.append({
        "cluster_id": f"test_{idx:05d}",
        "raw_index": int(idx),
        "documents": documents,
        "reference_summary": reference,
    })

if not summary_data:
    raise RuntimeError("No usable Multi-News test clusters were loaded.")

DATASET_PROVENANCE = {
    "dataset_repository": DATASET_REPO,
    "split": "test",
    "raw_split_rows": int(len(raw_test)),
    "usable_clusters": int(len(summary_data)),
    "excluded_unusable_rows": int(excluded),
    "first_cluster_id": summary_data[0]["cluster_id"],
    "last_cluster_id": summary_data[-1]["cluster_id"],
}
PROVENANCE_PATH.write_text(json.dumps(DATASET_PROVENANCE, indent=2), encoding="utf-8")
display(pd.DataFrame([DATASET_PROVENANCE]))

## Frozen Run82 summarizer

The following implementation preserves the method logic of the authoritative frozen Run82 pipeline: LDA topic relevance, FP-Growth lexical-pattern support, redundancy control, and deterministic sentence selection under the frozen parameters.

In [ ]:
import signal
from collections import Counter
import spacy

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth

# Exact experimental constants used by the finalized 120-scenario Round 1.
MIN_PATTERN_SUPPORT = 0.08
MIN_PATTERN_OCCURRENCES = 2
MAX_PATTERN_LENGTH = 3
MAX_TRANSACTION_ITEMS = 20
MAX_FREQUENT_ITEMSETS = 500_000
FP_TIMEOUT_SECONDS = 300
CANDIDATE_POOL_PER_TOPIC = 50
MAX_SENTENCES_FOR_SIMILARITY = 220
LDA_MAX_FEATURES = 5000
LDA_MAX_ITER = 10
LDA_N_JOBS = 1

# Parser active for sentence boundaries and dependency triples.
nlp_dep = spacy.load("en_core_web_sm", disable=["ner"])

# Round-1 lexical transaction pipeline: parser/NER disabled, tagger+lemmatizer retained.
nlp_lex = spacy.load("en_core_web_sm", disable=["parser", "ner"])
if "sentencizer" not in nlp_lex.pipe_names:
    nlp_lex.add_pipe("sentencizer")

def safe_sentencize(documents):
    if isinstance(documents, str):
        documents = [documents]
    sentences = []
    for document in documents:
        text = clean_text(document)
        if not text:
            continue
        doc = nlp_dep(text)
        for sent in doc.sents:
            s = clean_text(sent.text)
            if s:
                sentences.append(s)
    return sentences

def _normalise_rows(matrix):
    arr = np.asarray(matrix, dtype=float)
    denom = arr.sum(axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    return arr / denom

def fit_topic_model(sentences, n_topics=N_TOPICS, seed=SEED):
    n = len(sentences)
    if n == 0:
        return {
            "topic_score": np.array([], dtype=float),
            "theta": np.empty((0, 1), dtype=float),
            "phi": np.ones((1, 1), dtype=float),
            "vectorizer": None,
            "X": None,
            "feature_names": np.array(["fallback"], dtype=str),
            "dominant_topic": np.array([], dtype=int),
        }

    # Round-1 vectorizer contract.
    vectorizer = CountVectorizer(
        stop_words="english",
        lowercase=True,
        max_features=LDA_MAX_FEATURES,
        ngram_range=(1, 1),
        min_df=1,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
    )

    try:
        X = vectorizer.fit_transform(sentences)
    except ValueError:
        X = None

    if X is None or X.shape[1] == 0:
        return {
            "topic_score": np.ones(n, dtype=float),
            "theta": np.ones((n, 1), dtype=float),
            "phi": np.ones((1, 1), dtype=float),
            "vectorizer": None,
            "X": None,
            "feature_names": np.array(["fallback"], dtype=str),
            "dominant_topic": np.zeros(n, dtype=int),
        }

    effective_topics = max(
        1,
        min(int(n_topics), int(X.shape[0]), max(1, int(X.shape[1]))),
    )

    lda = LatentDirichletAllocation(
        n_components=effective_topics,
        random_state=int(seed),
        learning_method="batch",
        max_iter=LDA_MAX_ITER,
        n_jobs=LDA_N_JOBS,
    )
    theta = lda.fit_transform(X)
    phi = _normalise_rows(lda.components_)
    feature_names = vectorizer.get_feature_names_out()

    # Topic significance = mean strongest topic-word salience across unique words.
    X_bin = (X > 0).astype(float)
    word_topic_salience = phi.max(axis=0)
    raw_ts = np.asarray(X_bin @ word_topic_salience).ravel()
    token_counts = np.asarray(X_bin.sum(axis=1)).ravel()
    token_counts[token_counts == 0] = 1.0
    topic_score = raw_ts / token_counts
    dominant_topic = theta.argmax(axis=1)

    return {
        "topic_score": np.asarray(topic_score, dtype=float),
        "theta": np.asarray(theta, dtype=float),
        "phi": np.asarray(phi, dtype=float),
        "vectorizer": vectorizer,
        "X": X,
        "feature_names": np.asarray(feature_names),
        "dominant_topic": np.asarray(dominant_topic, dtype=int),
    }

def _topic_word_weight_map(topic_model):
    feature_names = topic_model.get("feature_names")
    phi = topic_model.get("phi")
    if feature_names is None or phi is None or len(feature_names) == 0:
        return {}
    strongest = np.asarray(phi).max(axis=0)
    return {
        str(term): float(weight)
        for term, weight in zip(feature_names, strongest)
    }

def lexical_transactions(sentences, batch_size=128):
    """Round-1 lemmatised lexical transactions."""
    transactions = []
    docs = nlp_lex.pipe(
        (str(sentence).lower() for sentence in sentences),
        batch_size=int(batch_size),
    )
    for doc in docs:
        tokens = []
        for token_obj in doc:
            token = (token_obj.lemma_ or token_obj.text).lower().strip()
            if (
                token_obj.is_alpha
                and not token_obj.is_stop
                and len(token) > 2
            ):
                tokens.append(token)
        transactions.append(sorted(set(tokens)))
    return transactions

class _FPGrowthTimeout(Exception):
    pass

def _fp_timeout_handler(signum, frame):
    raise _FPGrowthTimeout("FP-Growth exceeded timeout")

def pattern_scores(sentences, topic_model, min_support=MIN_PATTERN_SUPPORT):
    if not sentences:
        return np.array([], dtype=float), []

    raw_transactions = lexical_transactions(sentences)
    if not any(raw_transactions):
        return np.zeros(len(sentences), dtype=float), []

    topic_weight = _topic_word_weight_map(topic_model)

    # Round-1 deterministic topic-salience transaction pruning.
    transactions = []
    for items in raw_transactions:
        items = list(items)
        if len(items) > MAX_TRANSACTION_ITEMS:
            items = sorted(
                items,
                key=lambda term: (-topic_weight.get(term, 0.0), term),
            )[:MAX_TRANSACTION_ITEMS]
        transactions.append(sorted(set(items)))

    encoder = TransactionEncoder()
    encoded = encoder.fit(transactions).transform(transactions)
    frame = pd.DataFrame(encoded, columns=encoder.columns_)

    required_occurrences = min(
        int(MIN_PATTERN_OCCURRENCES),
        max(1, len(transactions)),
    )
    effective_support = max(
        float(min_support),
        float(required_occurrences) / max(1, len(transactions)),
    )

    previous_handler = signal.signal(signal.SIGALRM, _fp_timeout_handler)
    signal.alarm(int(FP_TIMEOUT_SECONDS))
    try:
        try:
            freq = fpgrowth(
                frame,
                min_support=effective_support,
                use_colnames=True,
                max_len=int(MAX_PATTERN_LENGTH),
            )
        except _FPGrowthTimeout as exc:
            raise RuntimeError(
                "FP-Growth timeout. No parameter was changed; "
                "the definitive run stopped explicitly."
            ) from exc
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, previous_handler)

    if freq.empty:
        return np.zeros(len(sentences), dtype=float), []

    # Round 1 stopped explicitly rather than truncating pathological output.
    if len(freq) > MAX_FREQUENT_ITEMSETS:
        raise RuntimeError(
            f"FP-Growth returned {len(freq):,} frequent itemsets, exceeding "
            f"the explicit cap {MAX_FREQUENT_ITEMSETS:,}. "
            "No parameter was changed."
        )

    column_index = {
        str(term): idx for idx, term in enumerate(encoder.columns_)
    }
    encoded_bool = np.asarray(encoded, dtype=bool)

    pattern_records = []
    scores = np.zeros(len(sentences), dtype=float)

    for row in freq.itertuples(index=False):
        itemset = frozenset(str(x) for x in row.itemsets)
        if not itemset:
            continue

        coherence = float(np.mean([
            topic_weight.get(term, 0.0) for term in itemset
        ]))
        support = float(row.support)
        weight = support * coherence

        pattern_records.append({
            "items": itemset,
            "support": support,
            "topic_coherence": coherence,
            "weight": weight,
        })

        columns = [
            column_index[term]
            for term in itemset
            if term in column_index
        ]
        if len(columns) != len(itemset):
            continue

        active = encoded_bool[:, columns].all(axis=1)
        scores[active] += weight

    return scores, pattern_records

def build_candidate_pool(topic_model, per_topic=CANDIDATE_POOL_PER_TOPIC):
    theta = np.asarray(topic_model.get("theta"))
    if theta.size == 0:
        return np.array([], dtype=int)

    selected = set()
    per_topic = max(1, int(per_topic))
    for topic_id in range(theta.shape[1]):
        order = np.argsort(-theta[:, topic_id])
        for idx in order[:per_topic]:
            selected.add(int(idx))
    return np.array(sorted(selected), dtype=int)

def _paper1_similarity(sentences, candidate_indices=None):
    """Round-1 candidate-only TF-IDF cosine matrix."""
    if candidate_indices is None:
        candidate_indices = list(range(len(sentences)))
    candidate_indices = list(candidate_indices)

    if not candidate_indices:
        return np.empty((0, 0)), candidate_indices

    if len(candidate_indices) > MAX_SENTENCES_FOR_SIMILARITY:
        candidate_indices = candidate_indices[:MAX_SENTENCES_FOR_SIMILARITY]

    subset = [sentences[i] for i in candidate_indices]

    if len(subset) == 1:
        return np.ones((1, 1), dtype=np.float32), candidate_indices

    try:
        X = TfidfVectorizer(
            stop_words="english",
            lowercase=True,
            token_pattern=r"(?u)\b\w\w+\b",
        ).fit_transform(subset)
        sim = cosine_similarity(
            X,
            dense_output=True,
        ).astype(np.float32, copy=False)
        return sim, candidate_indices
    except ValueError:
        return np.eye(len(subset), dtype=np.float32), candidate_indices

def generate_summary(documents):
    """
    Frozen Run-82 summariser with Round-1 method parity.

    Score(s) = alpha*TS(s) + beta*PR(s) - delta*Red(s,S)
    """
    sentences = safe_sentencize(documents)
    if not sentences:
        return "", {
            "n_sentences": 0,
            "patterns_found": 0,
            "candidate_count": 0,
        }

    target = max(1, min(int(SUMMARY_BUDGET), len(sentences)))

    # For short clusters the extractive summary is all available sentences.
    if len(sentences) <= target:
        return " ".join(sentences), {
            "n_sentences": int(len(sentences)),
            "patterns_found": 0,
            "candidate_count": int(len(sentences)),
        }

    topic_model = fit_topic_model(sentences, n_topics=N_TOPICS, seed=SEED)
    topic = np.asarray(topic_model["topic_score"], dtype=np.float32)
    patt, patterns = pattern_scores(
        sentences,
        topic_model=topic_model,
        min_support=MIN_PATTERN_SUPPORT,
    )
    patt = np.asarray(patt, dtype=np.float32)

    candidates = build_candidate_pool(
        topic_model,
        per_topic=CANDIDATE_POOL_PER_TOPIC,
    )
    if candidates.size == 0 or len(candidates) < target:
        candidates = np.arange(len(sentences), dtype=int)

    # CRITICAL Round-1 parity: rank by relevance BEFORE the 220-sentence cap.
    base = (
        ALPHA * topic
        + BETA * patt
    )
    ordered_candidates = sorted(
        (int(i) for i in candidates),
        key=lambda i: (-float(base[i]), int(i)),
    )

    sim, active_candidates = _paper1_similarity(
        sentences,
        ordered_candidates,
    )
    active_candidates = list(active_candidates)
    position = {
        idx: pos for pos, idx in enumerate(active_candidates)
    }

    selected = []
    remaining = set(active_candidates)

    while remaining and len(selected) < target:
        best_idx = None
        best_score = -np.inf

        for idx in sorted(remaining):
            if selected:
                i = position[idx]
                redundancy = max(
                    float(sim[i, position[j]])
                    for j in selected
                )
            else:
                redundancy = 0.0

            score = float(
                ALPHA * topic[idx]
                + BETA * patt[idx]
                - DELTA * redundancy
            )

            if (
                score > best_score
                or (
                    np.isclose(score, best_score)
                    and (best_idx is None or idx < best_idx)
                )
            ):
                best_score = score
                best_idx = int(idx)

        if best_idx is None:
            break

        selected.append(best_idx)
        remaining.remove(best_idx)

    # Extractive presentation follows original document order.
    summary = " ".join(
        sentences[i]
        for i in sorted(selected)
    )

    return summary, {
        "n_sentences": int(len(sentences)),
        "patterns_found": int(len(patterns)),
        "candidate_count": int(len(active_candidates)),
    }

print("Round-1 method-parity Run-82 summariser ready.")

## Round-2 preservation evaluator

The following functions reproduce the ROUGE/SU4 and dependency-triple preservation metrics used for the Round-2 method-parity evaluation. The later Round-3 exact-triple metric is a distinct metric object and is evaluated elsewhere in the repository.

In [ ]:
from rouge_score import rouge_scorer

# Round-1 ROUGE-LSum was scored separately on newline-separated sentences.
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)
rouge_lsum = rouge_scorer.RougeScorer(
    ["rougeLsum"],
    use_stemmer=True,
)

# Round-1 SU4 token contract.
SU4_TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def _rouge_tokens(text):
    return [
        token.lower()
        for token in SU4_TOKEN_RE.findall(str(text or ""))
    ]

def _skip_bigrams(tokens, max_skip=4):
    # Up to max_skip intervening tokens.
    for i in range(len(tokens)):
        upper = min(len(tokens), i + int(max_skip) + 2)
        for j in range(i + 1, upper):
            yield (tokens[i], tokens[j])

def _multiset_overlap(left, right):
    return sum((Counter(left) & Counter(right)).values())

def rouge_su4(reference, prediction, max_skip=4):
    ref_tokens = _rouge_tokens(reference)
    pred_tokens = _rouge_tokens(prediction)

    ref_units = (
        [("U", token) for token in ref_tokens]
        + [("B", a, b) for a, b in _skip_bigrams(ref_tokens, max_skip)]
    )
    pred_units = (
        [("U", token) for token in pred_tokens]
        + [("B", a, b) for a, b in _skip_bigrams(pred_tokens, max_skip)]
    )

    overlap = _multiset_overlap(ref_units, pred_units)
    precision = overlap / len(pred_units) if pred_units else 0.0
    recall = overlap / len(ref_units) if ref_units else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )
    return float(precision), float(recall), float(f1)

def _prepare_rouge_lsum(text):
    """Newline-separated sentences required by the Round-1 ROUGE-LSum evaluator."""
    text = str(text or "").strip()
    if not text:
        return ""

    chunks = []
    for paragraph in text.splitlines() or [text]:
        chunks.extend(
            re.split(r"(?<=[.!?])\s+", paragraph.strip())
        )
    return "\n".join(
        chunk.strip()
        for chunk in chunks
        if chunk.strip()
    )

def compute_all_rouge(reference, prediction):
    reference = str(reference or "")
    prediction = str(prediction or "")

    standard = rouge.score(reference, prediction)
    lsum = rouge_lsum.score(
        _prepare_rouge_lsum(reference),
        _prepare_rouge_lsum(prediction),
    )["rougeLsum"]
    su4_p, su4_r, su4_f = rouge_su4(
        reference,
        prediction,
        max_skip=4,
    )

    return {
        "rouge1_precision": float(standard["rouge1"].precision),
        "rouge1_recall": float(standard["rouge1"].recall),
        "rouge1_f1": float(standard["rouge1"].fmeasure),
        "rouge2_precision": float(standard["rouge2"].precision),
        "rouge2_recall": float(standard["rouge2"].recall),
        "rouge2_f1": float(standard["rouge2"].fmeasure),
        "rougeL_precision": float(standard["rougeL"].precision),
        "rougeL_recall": float(standard["rougeL"].recall),
        "rougeL_f1": float(standard["rougeL"].fmeasure),
        "rougeLsum_precision": float(lsum.precision),
        "rougeLsum_recall": float(lsum.recall),
        "rougeLsum_f1": float(lsum.fmeasure),
        "rougeSU4_precision": float(su4_p),
        "rougeSU4_recall": float(su4_r),
        "rougeSU4_f1": float(su4_f),
    }

def normalize_label(value):
    value = clean_text(value).lower()
    value = re.sub(r"[^a-z0-9_\s-]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def subtree_text(token):
    return clean_text(
        " ".join(
            t.text
            for t in sorted(token.subtree, key=lambda x: x.i)
        )
    )

def extract_dependency_triples(text):
    text = clean_text(text)
    if not text:
        return []

    doc = nlp_dep(text)
    candidates = []

    for sentence in doc.sents:
        roots = [
            token
            for token in sentence
            if token.dep_ == "ROOT"
            and token.pos_ in {"VERB", "AUX"}
        ]

        for root in roots:
            children = list(root.children)

            subjects = [
                token
                for token in children
                if token.dep_ in {"nsubj", "nsubjpass", "csubj"}
            ]

            objects = [
                token
                for token in children
                if token.dep_ in {"dobj", "obj", "attr", "oprd", "dative"}
            ]

            for subject in subjects:
                for obj in objects:
                    candidates.append((
                        subtree_text(subject),
                        root.lemma_ or root.text,
                        subtree_text(obj),
                        sentence.text,
                    ))

            for prep in [
                token for token in children if token.dep_ == "prep"
            ]:
                prep_objects = [
                    token
                    for token in prep.children
                    if token.dep_ in {"pobj", "obj"}
                ]
                for subject in subjects:
                    for obj in prep_objects:
                        predicate = (
                            f"{root.lemma_ or root.text}_"
                            f"{prep.lemma_ or prep.text}"
                        )
                        candidates.append((
                            subtree_text(subject),
                            predicate,
                            subtree_text(obj),
                            sentence.text,
                        ))

    output = []
    seen = set()

    for subject, predicate, obj, evidence in candidates:
        key = (
            normalize_label(subject),
            normalize_label(predicate),
            normalize_label(obj),
        )
        if all(key) and key not in seen:
            seen.add(key)
            output.append({
                "subject": clean_text(subject),
                "predicate": clean_text(predicate),
                "object": clean_text(obj),
                "evidence": clean_text(evidence),
                "method": "dependency",
            })

    return output

def canon_triple(triple):
    return (
        normalize_label(triple.get("subject", "")),
        normalize_label(triple.get("predicate", "")),
        normalize_label(triple.get("object", "")),
    )

def preservation_prf(predicted_triples, original_triples):
    pred = {canon_triple(x) for x in predicted_triples}
    gold = {canon_triple(x) for x in original_triples}
    pred.discard(("", "", ""))
    gold.discard(("", "", ""))

    tp = len(pred & gold)
    fp = len(pred - gold)
    fn = len(gold - pred)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )
    return (
        tp,
        fp,
        fn,
        float(precision),
        float(recall),
        float(f1),
    )

print("Round-1 method-parity evaluation utilities ready.")

In [ ]:
# Evaluate frozen Run82 on the held-out test split
# Public outputs intentionally exclude raw source text and generated/reference summaries.
from tqdm.auto import tqdm

rows = []
checkpoint_every = 50

if DETAIL_PATH.exists():
    prior = pd.read_csv(DETAIL_PATH)
    prior = prior.drop_duplicates("cluster_id", keep="last")
    completed_ids = set(prior["cluster_id"].astype(str))
    rows = prior.to_dict("records")
    print(f"Resuming from {len(completed_ids)} completed clusters.")
else:
    completed_ids = set()

for row in tqdm(summary_data, desc="Frozen Run82 test evaluation"):
    cluster_id = str(row["cluster_id"])
    if cluster_id in completed_ids:
        continue

    documents = row["documents"]
    reference = row["reference_summary"]
    original_text = clean_text(" ".join(documents))
    cluster_start = time.perf_counter()

    predicted, diagnostics = generate_summary(documents)
    rouge_metrics = compute_all_rouge(reference, predicted)

    original_triples = extract_dependency_triples(original_text)
    summary_triples = extract_dependency_triples(predicted)
    tp, fp, fn, p, r, f1 = preservation_prf(summary_triples, original_triples)

    original_words = len(original_text.split())
    summary_words = len(predicted.split())
    compression_ratio = summary_words / original_words if original_words else 0.0

    rec = {
        "cluster_id": cluster_id,
        "raw_index": int(row["raw_index"]),
        "dataset_split": "test",
        "run_order": 82,
        "alpha_topic": ALPHA,
        "beta_pattern": BETA,
        "delta_redundancy": DELTA,
        "summary_sentence_budget": SUMMARY_BUDGET,
        "n_topics": N_TOPICS,
        "source_sentence_count": int(diagnostics["n_sentences"]),
        "patterns_found": int(diagnostics["patterns_found"]),
        "candidate_count": int(diagnostics["candidate_count"]),
        **rouge_metrics,
        "original_words": int(original_words),
        "summary_words": int(summary_words),
        "compression_ratio": float(compression_ratio),
        "compression_gain": float(1.0 - compression_ratio),
        "original_triples": int(len(original_triples)),
        "summary_triples": int(len(summary_triples)),
        "matched_triples": int(tp),
        "unmatched_summary_triples": int(fp),
        "lost_original_triples": int(fn),
        "preservation_precision": float(p),
        "preservation_recall": float(r),
        "preservation_f1": float(f1),
        "cluster_runtime_seconds": float(time.perf_counter() - cluster_start),
    }
    rows.append(rec)
    completed_ids.add(cluster_id)

    if len(completed_ids) % checkpoint_every == 0:
        pd.DataFrame(rows).drop_duplicates("cluster_id", keep="last").sort_values(
            "raw_index", kind="stable"
        ).to_csv(DETAIL_PATH, index=False)
        gc.collect()

cluster_metrics = (
    pd.DataFrame(rows)
    .drop_duplicates("cluster_id", keep="last")
    .sort_values("raw_index", kind="stable")
    .reset_index(drop=True)
)
cluster_metrics.to_csv(DETAIL_PATH, index=False)

if len(cluster_metrics) != len(summary_data):
    raise RuntimeError(
        f"Evaluation incomplete: {len(cluster_metrics)}/{len(summary_data)} clusters."
    )

metric_cols = [
    "rouge1_precision", "rouge1_recall", "rouge1_f1",
    "rouge2_precision", "rouge2_recall", "rouge2_f1",
    "rougeL_precision", "rougeL_recall", "rougeL_f1",
    "rougeLsum_precision", "rougeLsum_recall", "rougeLsum_f1",
    "rougeSU4_precision", "rougeSU4_recall", "rougeSU4_f1",
    "preservation_precision", "preservation_recall", "preservation_f1",
    "compression_ratio", "compression_gain",
]
aggregates = {c: float(pd.to_numeric(cluster_metrics[c], errors="raise").mean()) for c in metric_cols}

result = {
    "run_order": 82,
    "dataset_split": "test",
    "alpha_topic": ALPHA,
    "beta_pattern": BETA,
    "delta_redundancy": DELTA,
    "n_topics": N_TOPICS,
    "summary_sentence_budget": SUMMARY_BUDGET,
    "historical_seed": SEED,
    "n_clusters": int(len(cluster_metrics)),
    **aggregates,
    "runtime_minutes": float(cluster_metrics["cluster_runtime_seconds"].sum() / 60.0),
    "status": "complete",
}

aggregate_result = pd.DataFrame([result])
aggregate_result.to_csv(RESULT_PATH, index=False)
display(aggregate_result)

In [ ]:
# Reproduction integrity checks and validation-to-test comparison
problems = []

if len(aggregate_result) != 1:
    problems.append("Expected one aggregate test row")
if int(aggregate_result.iloc[0]["run_order"]) != 82:
    problems.append("Aggregate result is not Run82")
if int(aggregate_result.iloc[0]["n_clusters"]) != len(summary_data):
    problems.append("Test cluster count does not match the loaded dataset")

if len(cluster_metrics) > 1 and np.allclose(
    cluster_metrics["rougeL_f1"].to_numpy(float),
    cluster_metrics["rougeLsum_f1"].to_numpy(float),
    rtol=0,
    atol=0,
):
    problems.append("ROUGE-LSum unexpectedly equals ROUGE-L for every cluster")

validation_metrics = FROZEN.get("selected_validation_metrics", {})
gap_rows = []
for metric in [
    "rouge1_f1", "rouge2_f1", "rougeL_f1", "rougeLsum_f1",
    "rougeSU4_recall", "rougeSU4_f1", "preservation_f1", "compression_gain",
]:
    if metric in validation_metrics and metric in aggregate_result.columns:
        v = float(validation_metrics[metric])
        t = float(aggregate_result.iloc[0][metric])
        gap_rows.append({
            "metric": metric,
            "validation": v,
            "test": t,
            "test_minus_validation": t - v,
            "absolute_gap": abs(t - v),
        })

gap_df = pd.DataFrame(gap_rows)
gap_df.to_csv(OUT / "run82_validation_vs_test.csv", index=False)

status = {
    "reproduction_complete": len(problems) == 0,
    "selected_validation_run_order": 82,
    "selection_frozen_before_test": bool(FROZEN.get("frozen_before_test", False)),
    "selection_split": FROZEN.get("selection_split"),
    "evaluation_split": "test",
    "frozen_parameters": EXPECTED,
    "historical_seed": 42,
    "validation_results_sha256": EXPECTED_SHA,
    "test_clusters": int(aggregate_result.iloc[0]["n_clusters"]),
    "dataset_repository": DATASET_REPO,
    "parameter_reselection_performed": False,
    "problems": problems,
}
STATUS_PATH.write_text(json.dumps(status, indent=2), encoding="utf-8")

if problems:
    raise RuntimeError("Reproduction integrity check failed:\n- " + "\n- ".join(problems))

print("Frozen Run82 reproduction: PASS")
display(gap_df)
print("Outputs:")
print(" -", RESULT_PATH)
print(" -", DETAIL_PATH)
print(" -", OUT / "run82_validation_vs_test.csv")
print(" -", STATUS_PATH)

## Public-output note

The notebook writes only numeric reproducibility outputs and dataset provenance. It does not redistribute Multi-News source articles, reference summaries, or generated summaries. Those texts remain available only in memory during execution and should be obtained from the official dataset source when reproducing the experiment.